# 01 — Detector Experiments

**Goal:** Understand the detector's behaviour across all three configurations,
analyse per-pattern hit rates, visualise category coverage, and identify
patterns that may need tuning.

## Overview
- Section 1: Environment setup and imports
- Section 2: Load the synthetic dataset
- Section 3: Run all three detector configurations
- Section 4: Per-category hit-rate analysis
- Section 5: Per-pattern coverage heatmap
- Section 6: Risk score distributions
- Section 7: Conclusions and tuning notes

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

from prompt_injection.detector import InjectionDetector, LogisticRegressionScorer
from prompt_injection.patterns import PATTERN_REGISTRY, list_categories
from prompt_injection.evaluation.dataset import SyntheticDataset

# ── Configuration ──────────────────────────────────────────────────────
SEED        = 42
THRESHOLD   = 0.50
FIGSIZE     = (12, 5)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print("Environment ready.")

## 2. Load the Synthetic Dataset

In [ ]:
ds = SyntheticDataset(n_injections=250, n_benign=250, seed=SEED).generate()
train_ds, test_ds = ds.train_test_split(test_size=0.20, seed=SEED)

print(f"Total records  : {len(ds)}")
print(f"Training split : {len(train_ds)}  (pos={train_ds.n_positive}, neg={train_ds.n_negative})")
print(f"Test split     : {len(test_ds)}   (pos={test_ds.n_positive}, neg={test_ds.n_negative})")
print(f"Categories     : {ds.categories()}")

## 3. Run All Three Detector Configurations

In [ ]:
# Fit classifier for Config C
clf = LogisticRegressionScorer()
clf.fit(train_ds.texts(), train_ds.labels())

detectors = {
    'A: Regex only':       InjectionDetector(mode='rules',  threshold=THRESHOLD),
    'B: Regex + Scoring':  InjectionDetector(mode='hybrid', threshold=THRESHOLD),
    'C: + Classifier':     InjectionDetector(mode='full',   threshold=THRESHOLD, classifier=clf),
}

# Scan all test records
results = {}
for name, det in detectors.items():
    records_results = []
    for record in test_ds:
        dr = det.scan(record.text, source_type=record.source_type or 'user')
        records_results.append({
            'id':           record.id,
            'label':        record.label,
            'is_injection': dr.is_injection,
            'risk_score':   dr.risk_score,
            'hit_count':    len(dr.hits),
            'categories':   dr.hit_categories,
        })
    results[name] = records_results

print("Detection complete.")
for name, res in results.items():
    flagged = sum(1 for r in res if r['is_injection'])
    print(f"  {name}: {flagged}/{len(res)} flagged")

## 4. Per-Category Hit-Rate Analysis

In [ ]:
from collections import defaultdict

categories = list_categories()
injection_records = [r for r in test_ds if r.label == 1]

# For Config B (hybrid) — richest information
det_b = detectors['B: Regex + Scoring']
cat_hits   = defaultdict(int)
cat_totals = defaultdict(int)

for record in injection_records:
    cat = record.attack_category or 'unknown'
    cat_totals[cat] += 1
    dr = det_b.scan(record.text, source_type=record.source_type or 'user')
    if dr.is_injection:
        cat_hits[cat] += 1

hit_rates = {cat: cat_hits[cat] / cat_totals[cat] for cat in categories if cat_totals[cat] > 0}

fig, ax = plt.subplots(figsize=FIGSIZE)
cats = list(hit_rates.keys())
rates = [hit_rates[c] * 100 for c in cats]
colors = ['#2ecc71' if r >= 80 else '#f39c12' if r >= 50 else '#e74c3c' for r in rates]
bars = ax.barh(cats, rates, color=colors, edgecolor='white', height=0.6)
ax.set_xlabel("Hit Rate (%)")
ax.set_title("Per-Category Detection Hit Rate — Config B (Hybrid)", fontweight='bold')
ax.set_xlim(0, 110)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, rate in zip(bars, rates):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{rate:.0f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig("category_hit_rates.png", dpi=130, bbox_inches='tight')
plt.show()
print("Hit rates by category:")
for cat, rate in sorted(hit_rates.items(), key=lambda x: -x[1]):
    print(f"  {cat:<30} {rate*100:.1f}%  ({cat_hits[cat]}/{cat_totals[cat]})")

## 5. Per-Pattern Coverage Table

In [ ]:
det_b = detectors['B: Regex + Scoring']
pattern_hits = defaultdict(int)

for record in injection_records:
    dr = det_b.scan(record.text, source_type=record.source_type or 'user')
    for hit in dr.hits:
        pattern_hits[hit.pattern_id] += 1

print(f"{'Pattern ID':<10} {'Category':<28} {'Sev':<8} {'Hits':>5}  Description")
print("-" * 90)
for entry in sorted(PATTERN_REGISTRY, key=lambda e: -pattern_hits.get(e['id'], 0)):
    hits = pattern_hits.get(entry['id'], 0)
    print(f"{entry['id']:<10} {entry['category']:<28} {entry['severity']:<8} {hits:>5}  {entry['description'][:45]}")

## 6. Risk Score Distributions

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
config_names = list(detectors.keys())

for ax, name in zip(axes, config_names):
    pos_scores = [r['risk_score'] for r in results[name] if r['label'] == 1]
    neg_scores = [r['risk_score'] for r in results[name] if r['label'] == 0]

    bins = np.linspace(0, 1, 25)
    ax.hist(neg_scores, bins=bins, alpha=0.65, color='#3498db', label='Benign',    edgecolor='white')
    ax.hist(pos_scores, bins=bins, alpha=0.65, color='#e74c3c', label='Injection', edgecolor='white')
    ax.axvline(THRESHOLD, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={THRESHOLD}')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel("Risk Score")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Count")
plt.suptitle("Risk Score Distributions by Configuration", fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("risk_score_distributions.png", dpi=130, bbox_inches='tight')
plt.show()

## 7. Conclusions and Tuning Notes

In [ ]:
print("=== Detector Experiment Summary ===")
print()
print("Pattern library:")
print(f"  Total patterns : {len(PATTERN_REGISTRY)}")
print(f"  Categories     : {len(list_categories())}")
print()
print("Tuning recommendations:")
print("  - Categories with < 70% hit rate may need additional patterns.")
print("  - Patterns with 0 hits on the test set are candidates for review.")
print("  - EVA patterns fire correctly but rely on scoring boost; verify threshold.")
print("  - Adjust threshold in Config B/C using Notebook 02 (policy evaluation).")